In [3]:
from train.train_lstm import train_lstm_pipeline
import numpy as np
import pandas as pd
import itertools
import json
import os
from pytorch_lightning.callbacks import ModelCheckpoint # Importa ModelCheckpoint

In [ ]:
ALL_VIDEOS = [
    # ('session0', 'center'),
    # ('session0', 'left'),
    # ('session0', 'right'),
    ('session1', 'center'),
    ('session1', 'left'),
    ('session1', 'right'),
    ('session2', 'center'),
    ('session2', 'left'),
    ('session2', 'right'),
    ('session3', 'center'),
    ('session3', 'left'),
    ('session3', 'right'),
    ('session4', 'center'),
    ('session4', 'left'),
    ('session4', 'right'),
    ('session5', 'center'),
    ('session5', 'left'),
    ('session5', 'right'),
    ('session6', 'center'),
    ('session6', 'left'),
    ('session6', 'right'),
]


# feature_path = 'faspect'
# feature_path = 'fbbox_angle'
# feature_path = 'fall'

In [ ]:
file = open('../../logs/fbbox/')

In [52]:
def get_metrics(param_grid, stage):
    summary_data = []
    for hs, nl, do, lr in itertools.product(*param_grid.values()):
        # print(f'**Grid Search**')
        # print(f"Entrenando con hs={hs}, nl={nl}, do={do}, lr={lr}")
        # print(f'Feature path: {feature_path}')
        exp_name = f'{feature_path}/hs{hs}_nl{nl}_do{do}_lr{lr}'
        metrics_file = f'../../logs/{exp_name}/version_0/metrics.csv'

        if os.path.exists(metrics_file):
            df_metrics = pd.read_csv(metrics_file)

            # Asegurarse de que 'val_loss' existe y no es nula
            if f'{stage}_loss' in df_metrics.columns and not df_metrics[f'{stage}_loss'].isnull().all():
                # Encontrar la fila con la mínima val_loss (excluyendo NaNs que ocurren en la última época)
                best_val_row = df_metrics.loc[df_metrics[f'{stage}_loss'].dropna().idxmin()]

                # Puedes agregar más métricas si son relevantes y están en tu CSV
                summary_data.append({
                    'hidden_size': hs,
                    'num_layers': nl,
                    'dropout': do,
                    'lr': lr,
                    'best_epoch': int(best_val_row['epoch']),
                    f'{stage}_loss': best_val_row[f'{stage}_loss'],
                    f'{stage}_mae': best_val_row[f'{stage}_mae'],
                    f'{stage}_mse': best_val_row[f'{stage}_mse'],
                    f'{stage}_acc': best_val_row[f'{stage}_acc'],
                })
        #     else:
        #         print(f"Advertencia: '{metrics_file}' no contiene 'val_loss' o está vacía.")
        # else:
        #     print(f"Advertencia: '{metrics_file}' no encontrado.")
    return pd.DataFrame(summary_data)


In [64]:
# search para features solo bbox x1, y1, x2, y2
feature_path = 'fbbox'
param_grid = {
    'hidden_size': [32, 64, 128],
    'num_layers': [1, 2, 3],
    'dropout': [0.0, 0.2, 0.5],
    'learning_rate': [1e-3, 1e-4, 1e-5],
}
stage = 'val'
summary_data = get_metrics(param_grid, stage)
df_summary = pd.DataFrame(summary_data)
df_summary = df_summary.sort_values(f"{stage}_loss").reset_index(drop=True)
df_summary.head(5)

,hidden_size,num_layers,dropout,lr,best_epoch,val_loss,val_mae,val_mse,val_acc
0,32,2,0.0,0.0010,26,12.408929,2.359870,14.792504,0.970037
1,32,2,0.0,0.0001,90,14.285874,2.342876,15.626202,0.970154
2,32,3,0.0,0.0010,45,14.706041,2.553959,16.011383,0.967367
3,32,3,0.2,0.0010,39,16.436113,2.508842,16.919725,0.967137
4,32,2,0.2,0.0010,45,19.857040,3.195739,21.672619,0.958833


In [ ]:
# search para features aspect -> bottom, aspect, angle, area
feature_path = 'faspect'
param_grid = {
    'hidden_size': [32, 64, 128],
    'num_layers': [1, 2, 3],
    'dropout': [0.0, 0.2, 0.5],
    'learning_rate': [1e-3, 1e-4, 1e-5],
}
stage = 'val'
summary_data = get_metrics(param_grid, stage)
df_summary = pd.DataFrame(summary_data)
df_summary = df_summary.sort_values(f"{stage}_loss").reset_index(drop=True)
df_summary.head(5)

,hidden_size,num_layers,dropout,lr,best_epoch,val_loss,val_mae,val_mse,val_acc
0,32,2,0.0,0.0010,47,8.891809,1.767065,11.678201,0.977224
1,32,3,0.0,0.0010,39,10.291531,1.982470,11.441545,0.974897
2,64,1,0.0,0.0010,48,16.263786,2.227581,15.172364,0.969937
3,32,1,0.0,0.0010,36,19.676395,2.537646,16.622574,0.967016
4,64,1,0.0,0.0001,92,15.084058,2.566622,19.571276,0.965988


In [ ]:
# search para features bbox_angle -> x1, y1, x2, y2, angle, area
feature_path = 'fbbox_angle'
param_grid = {
    'hidden_size': [32, 64, 128],
    'num_layers': [1, 2, 3],
    'dropout': [0.0, 0.1, 0.2],
    'learning_rate': [1e-2, 1e-3, 1e-4],
}
stage = 'val'
summary_data = get_metrics(param_grid, stage)
df_summary = pd.DataFrame(summary_data)
df_summary = df_summary.sort_values(f"{stage}_loss").reset_index(drop=True)
df_summary.head(5)

,hidden_size,num_layers,dropout,lr,best_epoch,val_loss,val_mae,val_mse,val_acc
0,64,3,0.0,0.0010,40,6.199358,1.584618,6.338459,0.979792
1,64,1,0.0,0.0010,41,7.958872,1.770978,10.546354,0.977542
2,64,2,0.0,0.0001,88,13.953409,1.912535,10.959879,0.975887
3,64,2,0.0,0.0010,36,12.166120,1.932660,9.896138,0.975458
4,64,1,0.1,0.0010,43,12.632465,2.102595,11.387042,0.973348


In [ ]:
# search para features fall -> todo menos width, height
feature_path = 'fall'
param_grid = {
    'hidden_size': [32, 64, 128],
    'num_layers': [1, 2, 3],
    'dropout': [0.0, 0.1, 0.2],
    'learning_rate': [1e-2, 1e-3, 1e-4],
}
stage = 'val'
summary_data = get_metrics(param_grid, stage)
df_summary = pd.DataFrame(summary_data)
df_summary = df_summary.sort_values(f"{stage}_loss").reset_index(drop=True)
df_summary.head(5)

,hidden_size,num_layers,dropout,lr,best_epoch,val_loss,val_mae,val_mse,val_acc
0,64,2,0.0,0.0010,31,8.051678,1.634077,7.315016,0.979319
1,64,2,0.0,0.0001,77,14.263813,1.708639,9.337360,0.978023
2,32,2,0.0,0.0010,58,9.081932,1.837727,8.515410,0.976290
3,64,3,0.0,0.0010,28,6.856952,1.825175,10.507398,0.975805
4,64,1,0.0,0.0010,35,9.610431,1.970258,9.876617,0.974472


### Mejor entrenamiento fuera del search

In [45]:
stage = 'val'
metrics_file = 'logs/tracking_lstm_sc/version_2/metrics.csv'
summary_data = []
if os.path.exists(metrics_file):
    df_metrics = pd.read_csv(metrics_file)
    if f'{stage}_loss' in df_metrics.columns and not df_metrics[f'{stage}_loss'].isnull().all():
        best_val_row = df_metrics.loc[df_metrics[f'{stage}_loss'].dropna().idxmin()]
        summary_data.append({
            'best_epoch': int(best_val_row['epoch']),
            f'{stage}_loss': best_val_row[f'{stage}_loss'],
            f'{stage}_mae': best_val_row[f'{stage}_mae'],
            f'{stage}_mse': best_val_row[f'{stage}_mse'],
            f'{stage}_acc': best_val_row[f'{stage}_acc'],
        })

df_summary = pd.DataFrame(summary_data)
df_summary = df_summary.sort_values(f"{stage}_loss").reset_index(drop=True)
display(df_summary)


,best_epoch,val_loss,val_mae,val_mse,val_acc
0,35,12.167837,1.722249,12.167837,0.977766
